In [3]:
import cv2
import os
import numpy as np
from sklearn import svm
import joblib

# 1. DEFINISCI L'HOG GLOBALE (fuori da ogni funzione)
# Questo oggetto hog sarà visibile ovunque nel file
win_size = (64, 128)
block_size = (16, 16)
block_stride = (8, 8)
cell_size = (8, 8)
nbins = 9
hog = cv2.HOGDescriptor(win_size, block_size, block_stride, cell_size, nbins)

def extract_hog(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img = cv2.resize(img, win_size)
    # 2. Usa l'oggetto hog definito sopra
    return hog.compute(img)

# Creazione dataset
X = []
labels = [] 

# Carica immagini persone
for f in os.listdir('persone'):
    feat = extract_hog('persone/' + f)
    if feat is not None:
        X.append(feat.flatten())
        labels.append(1)

# Carica immagini sfondo
for f in os.listdir('sfondo'):
    img = cv2.imread('sfondo/' + f, cv2.IMREAD_GRAYSCALE)
    if img is None: continue 
    
    # Controllo dimensione minima
    if img.shape[0] < 128 or img.shape[1] < 64: continue
    
    for _ in range(5): 
        y_coord = np.random.randint(0, img.shape[0]-128)
        x_coord = np.random.randint(0, img.shape[1]-64)
        patch = img[y_coord:y_coord+128, x_coord:x_coord+64]
        
        feat = hog.compute(patch)
        if feat is not None:
            X.append(feat.flatten())
            labels.append(0)

# Addestramento
clf = svm.SVC(kernel='linear', C=1.0)
clf.fit(X, labels)

# Salva il cervello
joblib.dump(clf, 'mio_detector_persone.pkl')
print("Training completato e modello salvato!")

ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
# 1. Carica l'immagine
path = 'persone/2026-06-30-154327_1.jpg' # Assicurati che questo nome sia ESATTAMENTE quello nel tuo PC
test_img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

# 2. CONTROLLO DI SICUREZZA
if test_img is None:
    print(f"Errore: Immagine non trovata in {path}")
    # Elenca i file nella cartella per vedere dove hai sbagliato
    import os
    print("File disponibili in 'persone/':", os.listdir('persone'))
else:
    # 3. Ora puoi procedere con sicurezza
    test_img_resized = cv2.resize(test_img, (64, 128))
    test_hog = hog.compute(test_img_resized).reshape(1, -1)
    print("Score sulla foto di training:", clf.decision_function(test_hog))

In [ ]:
import cv2
import joblib

# 1. Carica il tuo modello e inizializza l'HOG
clf = joblib.load('mio_detector_persone.pkl')
hog = cv2.HOGDescriptor((64, 128), (16, 16), (8, 8), (8, 8), 9)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.resize(frame, (640, 480))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. Sliding Window: dobbiamo scansionare l'immagine
    step_size = 32 # Sposta la finestra di 32 pixel alla volta
    window_size = (64, 128)
    max_score = -999
    for y in range(0, frame.shape[0] - window_size[1], step_size):
        for x in range(0, frame.shape[1] - window_size[0], step_size):
            # Estrai la patch
            patch = gray[y:y+window_size[1], x:x+window_size[0]]
            
            # Calcola HOG
            patch_hog = hog.compute(patch).reshape(1, -1)
            
            # Predizione della SVM
            prediction = clf.predict(patch_hog)
            score = clf.decision_function(patch_hog)
            if score > max_score:
                max_score = score
            
                if score > 0: 
                    cv2.rectangle(frame, (x, y), (x + window_size[0], y + window_size[1]), (0, 255, 0), 2)
                    print("Found")
                else:
                    print("Not found")
    
    cv2.imshow("Robot - Detector Custom", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
print(f"Score massimo nel frame: {max_score:.2f}")
cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def visualize_hog(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 128))
    
    # Parametri HOG
    win_size = (64, 128)
    cell_size = (8, 8)
    block_size = (16, 16)
    nbins = 9
    hog = cv2.HOGDescriptor(win_size, block_size, (8, 8), cell_size, nbins)
    
    # Calcoliamo i gradienti
    gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=1)
    mag, ang = cv2.cartToPolar(gx, gy)
    
    # Creiamo un'immagine vuota per visualizzare gli istogrammi
    vis = np.zeros(img.shape, dtype=np.float32)
    
    # Disegniamo una rappresentazione stilizzata dei gradienti
    for y in range(0, img.shape[0], cell_size[1]):
        for x in range(0, img.shape[1], cell_size[0]):
            cell_mag = mag[y:y+cell_size[1], x:x+cell_size[0]]
            cell_ang = ang[y:y+cell_size[1], x:x+cell_size[0]]
            
            # Istogramma della cella (9 bins)
            hist = np.zeros(nbins)
            for i in range(cell_size[1]):
                for j in range(cell_size[0]):
                    bin_idx = int((cell_ang[i, j] / np.pi) * nbins) % nbins
                    hist[bin_idx] += cell_mag[i, j]
            
            # Disegniamo la linea dominante nella cella
            max_bin = np.argmax(hist)
            angle = (max_bin / nbins) * np.pi
            cx, cy = x + cell_size[0]//2, y + cell_size[1]//2
            dx, dy = int(np.cos(angle)*10), int(np.sin(angle)*10)
            cv2.line(vis, (cx-dx, cy-dy), (cx+dx, cy+dy), (255), 1)

    plt.figure(figsize=(8, 6))
    plt.subplot(1, 2, 1); plt.imshow(img, cmap='gray'); plt.title("Originale")
    plt.subplot(1, 2, 2); plt.imshow(vis, cmap='gray'); plt.title("Visualizzazione Vettori HOG")
    plt.show()

# Esegui la visualizzazione su una tua foto
path = 'persone/2026-06-30-154327_1.jpg' 
visualize_hog(path)

[ WARN:0@27.507] global loadsave.cpp:278 findDecoder imread_('persone/2026-06-30-154327_1.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'
